In [ ]:
from pathlib import Path
import json
import sys



PROJECT_ROOT = Path.cwd().parent
PAPER_DIR = PROJECT_ROOT / "data" / "papers"

PROCESSED_DIR = PROJECT_ROOT /'data'/'processed'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SRC_DIR = PROJECT_ROOT / "src"

sys.path.append(str(SRC_DIR))

from paperscope.models import Chunk

chunks_path = PROCESSED_DIR / "chunks.json"

In [ ]:
with open(chunks_path, "r", encoding="utf-8") as f:
    chunk_data = json.load(f)

all_chunks = [Chunk(**item) for item in chunk_data]

print("Loaded chunks:", len(all_chunks))
print(all_chunks[0])

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device="cuda"
)

In [ ]:
sample_texts = [
    all_chunks[0].text,
    all_chunks[1].text,
    all_chunks[2].text
]

embeddings = embedding_model.encode(
    sample_texts,
    normalize_embeddings=True
)

In [ ]:
print("Chunks:", len(all_chunks))
print("Embedding matrix:", embeddings.shape)

In [ ]:
texts = [chunk.text for chunk in all_chunks]

embeddings = embedding_model.encode(
    texts,
    batch_size=8,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Chunks:", len(all_chunks))
print("Embedding matrix:", embeddings.shape)

In [ ]:
import numpy as np

EMBEDDINGS_PATH = PROJECT_ROOT / "data" / "processed" / "embeddings.npy"

np.save(EMBEDDINGS_PATH, embeddings)

print("Saved embeddings to:", EMBEDDINGS_PATH)

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

QDRANT_PATH = PROJECT_ROOT / "storage" / "qdrant"

client = QdrantClient(path=str(QDRANT_PATH))

In [ ]:
COLLECTION_NAME = "research_papers"

client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=1024,
        distance=Distance.COSINE
    )
)

In [ ]:
from qdrant_client.models import PointStruct

points = []

for i, (chunk, vector) in enumerate(zip(all_chunks, embeddings)):
    points.append(
        PointStruct(
            id=i,
            vector=vector.tolist(),
            payload={
                "chunk_id": chunk.chunk_id,
                "paper_id": chunk.paper_id,
                "title": chunk.title,
                "section_heading": chunk.section_heading,
                "section_number": chunk.section_number,
                "page_start": chunk.page_start,
                "page_end": chunk.page_end,
                "chunk_index": chunk.chunk_index,
                "text": chunk.text
            }
        )
    )

client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)

In [ ]:
info = client.get_collection(COLLECTION_NAME)

print(info)

In [ ]:
query = "How does Self-RAG decide when retrieval is necessary?"

query_vector = embedding_model.encode(
    query,
    normalize_embeddings=True
)

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector.tolist(),
    limit=5
)

for rank, point in enumerate(results.points, start=1):
    print(f"Rank: {rank}")
    print(f"Score: {point.score:.4f}")
    print(f"Paper: {point.payload['title']}")
    print(f"Section: {point.payload['section_heading']}")
    print(f"Pages: {point.payload['page_start']} - {point.payload['page_end']}")
    print(f"Text: {point.payload['text'][:500]}")
    print("-" * 100)

In [ ]:
def dense_retrieve(
    query: str,
    embedding_model,
    client,
    collection_name: str,
    top_k: int = 5
):
    query_vector = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    results = client.query_points(
        collection_name=collection_name,
        query=query_vector.tolist(),
        limit=top_k
    )

    return results.points

In [ ]:
results = dense_retrieve(
    query="How does Self-RAG decide when retrieval is necessary?",
    embedding_model=embedding_model,
    client=client,
    collection_name=COLLECTION_NAME,
    top_k=5
)

In [ ]:
test_queries = [
    "How does Self-RAG decide when retrieval is necessary?",
    "What is late interaction in ColBERT?",
    "How does HyDE improve zero-shot dense retrieval?",
    "What problem does RAPTOR solve?",
    "How does CRAG handle poor retrieval results?",
    "What does Lost in the Middle say about long context?"
]



for query in test_queries:
    print("\nQUERY:", query)
    print("=" * 100)

    results = dense_retrieve(
        query=query,
        embedding_model=embedding_model,
        client=client,
        collection_name=COLLECTION_NAME,
        top_k=3
    )

    for rank, point in enumerate(results, start=1):
        print(
            rank,
            round(point.score, 4),
            "|",
            point.payload["title"],
            "|",
            point.payload["section_heading"]
        )

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [
    chunk.text.lower().split()
    for chunk in all_chunks
]

bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
def bm25_retrieve(
    query: str,
    bm25,
    chunks,
    top_k: int = 5
):
    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)

    ranked_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]

    results = []

    for index in ranked_indices:
        results.append({
            "score": scores[index],
            "chunk": chunks[index]
        })

    return results

In [ ]:
results = bm25_retrieve(
    query="How does Self-RAG decide when retrieval is necessary?",
    bm25=bm25,
    chunks=all_chunks,
    top_k=5
)

In [ ]:
for rank, result in enumerate(results, start=1):
    chunk = result["chunk"]

    print(
        rank,
        round(result["score"], 4),
        "|",
        chunk.title,
        "|",
        chunk.section_heading
    )

In [ ]:
for query in test_queries:
    print("\nQUERY:", query)
    print("=" * 100)

    print("\nDENSE")
    dense_results = dense_retrieve(
        query=query,
        embedding_model=embedding_model,
        client=client,
        collection_name=COLLECTION_NAME,
        top_k=3
    )

    for rank, point in enumerate(dense_results, start=1):
        print(
            rank,
            round(point.score, 4),
            "|",
            point.payload["title"],
            "|",
            point.payload["section_heading"]
        )

    print("\nBM25")
    bm25_results = bm25_retrieve(
        query=query,
        bm25=bm25,
        chunks=all_chunks,
        top_k=3
    )

    for rank, result in enumerate(bm25_results, start=1):
        chunk = result["chunk"]

        print(
            rank,
            round(result["score"], 4),
            "|",
            chunk.title,
            "|",
            chunk.section_heading
        )

- Dense retrieval is usually better at semantic meaning.
- BM25 is usually better when the query contains exact technical terms, names, datasets, acronyms, or unusual phrases.

In [ ]:
def reciprocal_rank_fusion(
    dense_results,
    bm25_results,
    k: int = 60
):
    scores = {}
    chunks_by_id = {}

    for rank, point in enumerate(dense_results, start=1):
        chunk_id = point.payload["chunk_id"]

        scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (k + rank)
        chunks_by_id[chunk_id] = point.payload

    for rank, result in enumerate(bm25_results, start=1):
        chunk = result["chunk"]
        chunk_id = chunk.chunk_id

        scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (k + rank)

        chunks_by_id[chunk_id] = {
            "chunk_id": chunk.chunk_id,
            "paper_id": chunk.paper_id,
            "title": chunk.title,
            "section_heading": chunk.section_heading,
            "section_number": chunk.section_number,
            "page_start": chunk.page_start,
            "page_end": chunk.page_end,
            "chunk_index": chunk.chunk_index,
            "text": chunk.text
        }

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        {
            "rrf_score": score,
            "chunk": chunks_by_id[chunk_id]
        }
        for chunk_id, score in ranked
    ]

In [ ]:
def hybrid_retrieve(
    query,
    embedding_model,
    client,
    collection_name,
    bm25,
    chunks,
    candidate_k=10,
    top_k=5
):
    dense_results = dense_retrieve(
        query=query,
        embedding_model=embedding_model,
        client=client,
        collection_name=collection_name,
        top_k=candidate_k
    )

    bm25_results = bm25_retrieve(
        query=query,
        bm25=bm25,
        chunks=chunks,
        top_k=candidate_k
    )

    fused_results = reciprocal_rank_fusion(
        dense_results,
        bm25_results
    )

    return fused_results[:top_k]

In [ ]:
results = hybrid_retrieve(
    query="How does Self-RAG decide when retrieval is necessary?",
    embedding_model=embedding_model,
    client=client,
    collection_name=COLLECTION_NAME,
    bm25=bm25,
    chunks=all_chunks,
    candidate_k=10,
    top_k=5
)

In [ ]:
for rank, result in enumerate(results, start=1):
    chunk = result["chunk"]

    print(
        rank,
        round(result["rrf_score"], 5),
        "|",
        chunk["title"],
        "|",
        chunk["section_heading"]
    )

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "Qwen/Qwen3-Reranker-0.6B",
    device="cuda"
)

In [ ]:
candidates = hybrid_retrieve(
    query="How does Self-RAG decide when retrieval is necessary?",
    embedding_model=embedding_model,
    client=client,
    collection_name=COLLECTION_NAME,
    bm25=bm25,
    chunks=all_chunks,
    candidate_k=15,
    top_k=15
)

In [ ]:
pairs = [
    (
        "How does Self-RAG decide when retrieval is necessary?",
        result["chunk"]["text"]
    )
    for result in candidates
]

In [ ]:
rerank_scores = reranker.predict(pairs)

reranked = []

for result, score in zip(candidates, rerank_scores):
    reranked.append({
        "rerank_score": float(score),
        "chunk": result["chunk"]
    })

reranked.sort(
    key=lambda x: x["rerank_score"],
    reverse=True
)

In [ ]:
for rank, result in enumerate(reranked[:5], start=1):
    chunk = result["chunk"]

    print(
        rank,
        round(result["rerank_score"], 4),
        "|",
        chunk["title"],
        "|",
        chunk["section_heading"]
    )

In [ ]:
def reranked_retrieve(
    query,
    embedding_model,
    client,
    collection_name,
    bm25,
    chunks,
    reranker,
    candidate_k=15,
    top_k=5
):
    candidates = hybrid_retrieve(
        query=query,
        embedding_model=embedding_model,
        client=client,
        collection_name=collection_name,
        bm25=bm25,
        chunks=chunks,
        candidate_k=candidate_k,
        top_k=candidate_k
    )

    pairs = [
        (query, result["chunk"]["text"])
        for result in candidates
    ]

    scores = reranker.predict(pairs)

    reranked = []

    for result, score in zip(candidates, scores):
        reranked.append({
            "rerank_score": float(score),
            "chunk": result["chunk"]
        })

    reranked.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked[:top_k]

In [ ]:
results = reranked_retrieve(
    query="How does Self-RAG decide when retrieval is necessary?",
    embedding_model=embedding_model,
    client=client,
    collection_name=COLLECTION_NAME,
    bm25=bm25,
    chunks=all_chunks,
    reranker=reranker,
    candidate_k=15,
    top_k=5
)

In [ ]:
for rank, result in enumerate(results, start=1):
    chunk = result["chunk"]

    print(
        rank,
        round(result["rerank_score"], 4),
        "|",
        chunk["title"],
        "|",
        chunk["section_heading"]
    )